# How much does the regression recipe set S?

The synthesis estimates the susceptibility $S(D) = \mathrm{d}\log N_d / \mathrm{d}\log N(>D)$ with one recipe:
log10 both variables, sort by $\log N(>D)$, ~15 hours per equal-number bin (5–20 bins), regress the **bin medians** with OLS.
This notebook re-fits every synthesis source (same files, same screens) with estimators that do not bin, and with
bivariate least squares that allows for error in both variables:

| estimator | what it minimises | assumption about errors |
|---|---|---|
| **binned medians** (current) | OLS through bin medians | none explicit; bins are sorted on the noisy x |
| **OLS** y on x | vertical residuals | x error-free → slope biased low (regression dilution) |
| **Theil–Sen** | median of pairwise slopes | robust version of OLS; still diluted by x error |
| **inverse OLS** (x on y, inverted) | horizontal residuals | y error-free → upper bound |
| **RMA** / geometric mean | triangle areas | error variance ratio = total variance ratio |
| **orthogonal** (major axis) | perpendicular distances | equal errors in log x and log y (λ = 1) |
| **Deming** | weighted perpendicular | λ = σ²(log Nd error) / σ²(log N error), set per method below |
| **York** (100 nm only) | per-point weighted | per-hour Nd error where the file has one (NDROP), else as Deming |

All of the moment estimators are fixed by three numbers: $s_x$, $s_y$ and the correlation $r$.
OLS $= r\,s_y/s_x$, RMA $= s_y/s_x$, inverse $= s_y/(r\,s_x)$, so **OLS/RMA = |r|** and inverse/RMA = 1/|r|. With
the hourly correlations of 0.3–0.7 seen in the quicklooks, the estimators can differ by a factor of 2–3; Deming and York
sit between OLS (λ → ∞) and inverse OLS (λ → 0).

Binning on x does not remove regression dilution: the hours are sorted on the *measured* $\log N$, so each bin's
median x still carries its noise and the bin medians of y regress towards the mean in the same way as OLS. Binning
mainly reduces the scatter about the line (r through bin medians is high), not the bias of the slope.

The error ratio λ is the weak point of every errors-in-variables fit: "error" in x here is not only the
SMPS/DMPS counting error but everything that decorrelates surface or below-cloud N(>D) from the aerosol that
actually activated (vertical mixing, entrainment, the hour average). Section 6 therefore shows S over a range of λ.

D$_\Delta$ (the zero crossing of mean log Nd − mean log N(>D)) does not depend on the regression, so S(D$_\Delta$) for
each estimator is that estimator's S(D) curve evaluated at the same D$_\Delta$.

Run from the repository root on Arrhenius (`conda activate analysis`). About 1–3 min per 1000 hours of data for the bootstrap.

## 1. Settings

In [ ]:
import glob
import os
import re
from types import SimpleNamespace

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import theilslopes

import arm_ena_nd_pnsd as ae
from plot_s_at_ddelta import SOURCES, SKIP, REG_COL, MK, QA, crossing, at

RESULTS = "results"                        # the quicklook tables and built files (results/**)
FIG_DIR = "figures/estimators"
OUT_CSV = "results/s_estimators.csv"
N_BOOT = 300                               # day-block resamples (same resamples as the quicklook: seed 0)
D_REF = 100.0                              # nm
EXCLUDE = {"Schneefernerhaus", "MOSAiC (Cloudnet, SMPS)", "Bankhead", "EPCAPE Mt Soledad"}   # as in model_obs_s_dsb
MERGE_CAO = True                           # plot Marine CAO as Marine
SEL = ["ACE-ENA", "ACTIVATE summer", "SGP", "EPCAPE pier", "Hyytiala", "Puijo"]               # S(D) curves, section 8

# Assumed 1-sigma errors in log10 units for Deming / York. sigma_x covers the size-distribution counting error plus the
# mismatch between measured and activating aerosol at hourly resolution; it is a judgement, so section 6 sweeps lambda.
SIG_Y = {"Aircraft": 0.05, "Ground in situ": 0.05, "Ship in situ": 0.08, "Ground NDROP": 0.15, "Cloudnet": 0.25}
SIG_X = {"Aircraft": 0.05, "Ground in situ": 0.10, "Ship in situ": 0.10, "Ground NDROP": 0.10, "Cloudnet": 0.10}
LAMBDAS = np.array([0.05, 0.1, 0.25, 0.5, 1, 2, 4, 10, 20, 100])
NB_LIST = [5, 10, 20, 40, 80]              # bin counts for the binning sensitivity (section 7)
os.makedirs(FIG_DIR, exist_ok=True)

## 2. Sources, files and screens

Same source list and instrument choice as `plot_s_at_ddelta.py` (SOURCES, parked UHSAS rows skipped), minus `EXCLUDE`.
The built file is the quicklook table's stem (`<file>[_tag]_quicklook.csv` → `<file>.nc`). Screens are read from the
quicklook text output where it exists, otherwise the synthesis defaults are used (cloud fraction ≥ 0.5; Cloudnet
base ≤ 2000 m and top ≤ 3000 m, AO2018 Cloudnet 1000/2000 m; `_relerr` tables: Nd error/Nd ≤ 0.5). Each source is
checked against its quicklook table: the hour count at 100 nm and the binned S(100 nm) must reproduce it (`check`).

In [ ]:
def source_list():
    out, seen = [], set()
    for csv in sorted(glob.glob(f"{RESULTS}/**/*_quicklook.csv", recursive=True)):
        p = csv.replace("\\", "/")
        if any(re.search(s, p) for s in SKIP):
            continue
        src = next((s for s in SOURCES if re.search(s[0], p)), None)
        if src is None or src[4] == "parked" or src[1] in EXCLUDE or src[1] in seen:
            continue
        seen.add(src[1])
        out.append(dict(csv=p, label=src[1], regime=src[2], method=src[3], quality=src[4]))
    return out


def built_file(csv):
    stem = csv[: -len("_quicklook.csv")]
    for c in (stem, re.sub(r"_(relerr|sza\d+)$", "", stem), re.sub(r"_relerr(_sza\d+)?$", "", stem)):
        if os.path.exists(c + ".nc"):
            return c + ".nc"
    return None


def _ns(**kw):
    ns = SimpleNamespace(max_rel_err=None, min_lwp=None, max_cloud_base=None, max_cloud_top=None, max_sza=None,
                         coupled_only=False, dz_coupled=150.0, latlon=None, min_cloud_frac=0.5)
    ns.__dict__.update(kw)
    return ns


def screen_options(csv, method):
    """Candidate screens: from the quicklook text (if any), then the synthesis defaults."""
    d = {}
    if method == "Cloudnet":
        d.update(max_cloud_base=1000, max_cloud_top=2000) if "ao2018" in csv else d.update(max_cloud_base=2000, max_cloud_top=3000)
    if "_relerr" in csv:
        d["max_rel_err"] = 0.5
    cands = []
    txt = csv[:-4] + ".txt"
    if os.path.exists(txt):
        s = open(txt, encoding="utf-8", errors="replace").read()
        mcf = re.search(r"Using cloud_frac >= ([\d.]+)", s)
        if mcf:
            scr = (re.search(r"Screens applied to every table: (.*)", s) or [None, ""])[1]
            t = {"min_cloud_frac": float(mcf.group(1))}
            for pat, k in [(r"Nd error/Nd <= ([\d.]+)", "max_rel_err"), (r"LWP >= ([\d.]+)", "min_lwp"),
                           (r"cloud base <= ([\d.]+) m", "max_cloud_base"), (r"cloud top <= ([\d.]+) m", "max_cloud_top")]:
                mm = re.search(pat, scr)
                if mm:
                    t[k] = float(mm.group(1))
            cands.append(("txt", _ns(**t)))
    cands.append(("default", _ns(**d)))
    return cands


def load(src):
    """Screened hourly arrays for one source, reproducing its quicklook selection."""
    nc = built_file(src["csv"])
    if nc is None:
        return None, "no built .nc next to the quicklook table"
    ql = pd.read_csv(src["csv"])
    n_ql = int(ql.n.iloc[np.argmin(np.abs(ql.D_nm.to_numpy() - D_REF))])
    ds = xr.open_dataset(nc)
    diam = ds["diameter"].values
    i100 = int(np.argmin(np.abs(diam - D_REF)))
    for how, ns in screen_options(src["csv"], src["method"]):
        try:
            base, cf, notes = ae._base_mask(ds, ns)
        except SystemExit:
            continue
        m = base & (cf >= ns.min_cloud_frac)
        ccn, y = ds["CCN"].values[m], ds["CDNC"].values[m]
        n100 = int((np.isfinite(ccn[:, i100]) & np.isfinite(y) & (ccn[:, i100] > 0) & (y > 0)).sum())
        if n100 == n_ql:
            break
    err = None
    if "drop_number_conc_toterror" in ds:
        err = ds["drop_number_conc_toterror"].values[m] / ds["CDNC"].values[m]
    out = dict(nc=nc, diam=diam, i100=i100, ccn=ccn, y=y, t=ds["time"].values[m], relerr=err, ql=ql,
               screens=f"{how}: cf>={ns.min_cloud_frac:g}" + ("; " + "; ".join(notes) if notes else ""),
               hours_match=n100 == n_ql, n_ql=n_ql)
    ds.close()
    return out, None


srcs = source_list()
print(f"{len(srcs)} sources")

## 3. Estimators

Every moment estimator is computed from per-day sums (n, Σx, Σy, Σx², Σy², Σxy), so a day-block bootstrap is a
matrix product of resample counts with those sums. The resamples are drawn exactly as in `bootstrap_by_day` (seed 0),
so all estimators – including the binned one – see the same resampled days and their differences can be bootstrapped too.

In [ ]:
def day_codes(t):
    day = pd.DatetimeIndex(t).normalize().to_numpy()
    days = np.unique(day)
    return np.searchsorted(days, day), len(days)


def boot_counts(G, n_boot, seed=0):
    """Day counts per resample, the same draws as ae.bootstrap_by_day (rng.integers(0, G, G) per resample)."""
    rng = np.random.default_rng(seed)
    W = np.empty((n_boot, G))
    for b in range(n_boot):
        W[b] = np.bincount(rng.integers(0, G, G), minlength=G)
    return W


def log_pairs(ccn, y):
    ly = np.where(np.isfinite(y) & (y > 0), np.log10(np.where(y > 0, y, 1)), np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        lx = np.where(np.isfinite(ccn) & (ccn > 0) & np.isfinite(ly)[:, None], np.log10(ccn), np.nan)
    return lx, ly


def day_sums(lx, ly, code, G):
    """(6, G, C): n, Σx, Σy, Σxx, Σyy, Σxy per day and cutoff over valid pairs."""
    C = lx.shape[1]
    S = np.zeros((6, G, C))
    for c in range(C):
        v = np.isfinite(lx[:, c])
        g, x, yy = code[v], lx[v, c], ly[v]
        for k, w in enumerate((np.ones_like(x), x, yy, x * x, yy * yy, x * yy)):
            S[k, :, c] = np.bincount(g, w, minlength=G)
    return S


def moment_slopes(S, W, lambdas):
    """Slopes for weights W (B, G). Returns dict of (B, C) arrays; 'dem' is (L, B, C)."""
    n, sx, sy, sxx, syy, sxy = (W @ S[k] for k in range(6))
    with np.errstate(divide="ignore", invalid="ignore"):
        n = np.where(n >= 10, n, np.nan)
        Sxx, Syy, Sxy = sxx - sx * sx / n, syy - sy * sy / n, sxy - sx * sy / n
        out = dict(ols=Sxy / Sxx, inv=Syy / Sxy, rma=np.sign(Sxy) * np.sqrt(Syy / Sxx), r=Sxy / np.sqrt(Sxx * Syy), n=n)
        lam = np.asarray(lambdas, float)[:, None, None]
        a = Syy - lam * Sxx
        out["dem"] = (a + np.sqrt(a * a + 4 * lam * Sxy * Sxy)) / (2 * Sxy)
    out["orth"] = out["dem"][list(lambdas).index(1.0)] if 1.0 in list(lambdas) else np.nan
    return out


def deming(S, W, lam):
    return moment_slopes(S, W, [lam])["dem"][0]


def york(x, y, sx, sy, n_iter=100, tol=1e-9):
    """York et al. (2004) bivariate least squares, uncorrelated errors; sx, sy scalars or per-point arrays."""
    wx, wy = 1 / np.broadcast_to(sx, x.shape) ** 2, 1 / np.broadcast_to(sy, y.shape) ** 2
    b = np.polyfit(x, y, 1)[0]
    for _ in range(n_iter):
        Wt = wx * wy / (wx + b * b * wy)
        U, V = x - (Wt * x).sum() / Wt.sum(), y - (Wt * y).sum() / Wt.sum()
        beta = Wt * (U / wy + b * V / wx)
        bn = (Wt * beta * V).sum() / (Wt * beta * U).sum()
        if abs(bn - b) < tol:
            return bn
        b = bn
    return b


def binned(lx, ly, nb, by="x", stat=np.median):
    """Equal-number bins on x (current recipe) or on y; OLS through the bin statistics."""
    ok = np.isfinite(lx) & np.isfinite(ly)
    x, y = lx[ok], ly[ok]
    if x.size < 2 * nb:
        return np.nan
    o = np.argsort(x if by == "x" else y)
    xb = np.array([stat(c) for c in np.array_split(x[o], nb)])
    yb = np.array([stat(c) for c in np.array_split(y[o], nb)])
    return np.polyfit(xb, yb, 1)[0] if np.ptp(xb) > 0 else np.nan


ci = lambda a: tuple(np.nanpercentile(a, [2.5, 97.5])) if np.isfinite(a).sum() >= 20 else (np.nan, np.nan)

# Self-test: York with constant errors equals Deming with lambda = sy^2/sx^2; OLS/RMA = |r|.
_r = np.random.default_rng(3); _xt = _r.normal(2.5, 0.3, 2000)
_x, _y = _xt + _r.normal(0, 0.1, 2000), 0.8 * _xt + _r.normal(0, 0.15, 2000)
_S = day_sums(_x[:, None], _y, np.arange(2000) % 50, 50)
_m = moment_slopes(_S, np.ones((1, 50)), [2.25])
assert abs(york(_x, _y, 0.1, 0.15) - _m["dem"][0, 0, 0]) < 1e-6
assert abs(_m["ols"][0, 0] / _m["rma"][0, 0] - abs(_m["r"][0, 0])) < 1e-12
print(f"self-test ok (true 0.80: OLS {_m['ols'][0,0]:.2f}, RMA {_m['rma'][0,0]:.2f}, Deming {_m['dem'][0,0,0]:.2f})")

## 4. Fit every source

In [ ]:
rows, curves, sweep, binsens = [], {}, {}, {}
for src in srcs:
    d, why = load(src)
    if d is None:
        print(f"  {src['label']:26s} skipped: {why}")
        continue
    diam, i100, ccn, y, t = d["diam"], d["i100"], d["ccn"], d["y"], d["t"]
    lx, ly = log_pairs(ccn, y)
    code_, G = day_codes(t)
    S = day_sums(lx, ly, code_, G)
    W = boot_counts(G, N_BOOT)
    lam0 = (SIG_Y[src["method"]] / SIG_X[src["method"]]) ** 2
    lams = sorted(set(LAMBDAS.tolist()) | {1.0, lam0})
    full, bt = moment_slopes(S, np.ones((1, G)), lams), moment_slopes(S, W, lams)
    il0 = lams.index(lam0)

    # current recipe, its bootstrap on the same resamples, D_Delta
    tab, dd = ae.susceptibility_table(ccn, y, diam)
    sb_bin, dd_boot = ae.bootstrap_by_day(t, ccn, y, diam, N_BOOT)
    dd_ql = crossing(d["ql"].D_nm.to_numpy(float), d["ql"].delta_dex.to_numpy(float))

    # 100 nm extras: Theil-Sen (subsampled to <= 3000 h) and York with per-hour Nd errors where available
    x1, y1 = lx[:, i100], ly
    v = np.isfinite(x1)
    rng = np.random.default_rng(1)
    sub = rng.choice(np.where(v)[0], min(v.sum(), 3000), replace=False)
    ts = theilslopes(y1[sub], x1[sub])[0]
    sy_pt = np.full(v.sum(), SIG_Y[src["method"]])
    if d["relerr"] is not None:
        re_ = d["relerr"][v]
        sy_pt = np.where(np.isfinite(re_) & (re_ > 0), np.clip(re_ / np.log(10), 0.02, None), sy_pt)
    xv, yv, cv = x1[v], y1[v], code_[v]
    groups = [np.where(cv == g)[0] for g in range(G)]
    yk = york(xv, yv, SIG_X[src["method"]], sy_pt)
    yk_b = []
    for b in range(N_BOOT):
        idx = np.concatenate([np.repeat(groups[g], int(k)) for g, k in enumerate(W[b]) if k])
        yk_b.append(york(xv[idx], yv[idx], SIG_X[src["method"]], sy_pt[idx]))
    yk_b = np.array(yk_b)

    est = {"binned": (tab.S.to_numpy(), sb_bin), "ols": (full["ols"][0], bt["ols"]), "rma": (full["rma"][0], bt["rma"]),
           "orth": (full["orth"][0], bt["orth"]), "inv": (full["inv"][0], bt["inv"]),
           "deming": (full["dem"][il0, 0], bt["dem"][il0])}
    curves[src["label"]] = dict(diam=diam, dd=dd_ql, est=est, r=full["r"][0])
    sweep[src["label"]] = (np.array(lams), full["dem"][:, 0, i100], lam0)
    binsens[src["label"]] = {by: [binned(x1, y1, nb, by) for nb in NB_LIST] for by in ("x", "y")}
    binsens[src["label"]]["xmean"] = [binned(x1, y1, nb, "x", np.mean) for nb in NB_LIST]

    row = dict(label=src["label"], regime=src["regime"], method=src["method"], quality=src["quality"],
               hours=int(v.sum()), days=G, r100=full["r"][0, i100], lam=lam0, D_Delta=dd_ql,
               check="ok" if d["hours_match"] and abs(tab.S[i100] - at(d["ql"].D_nm.to_numpy(float), d["ql"].S.to_numpy(float), diam[i100])) < 0.01
               else f"hours {v.sum()} vs {d['n_ql']} in table", screens=d["screens"])
    for k, (pt, bs) in est.items():
        row[f"{k}_100"] = pt[i100]
        row[f"{k}_100_lo"], row[f"{k}_100_hi"] = ci(bs[:, i100])
        if np.isfinite(dd_ql):
            row[f"{k}_DD"] = at(diam, pt, dd_ql)
            row[f"{k}_DD_lo"], row[f"{k}_DD_hi"] = ci(np.array([at(diam, s, dd_ql) for s in bs]))
    for k in ("ols", "deming"):                                    # paired difference to the current recipe
        row[f"{k}_minus_binned_lo"], row[f"{k}_minus_binned_hi"] = ci(est[k][1][:, i100] - sb_bin[:, i100])
    row.update(theilsen_100=ts, york_100=yk, york_100_lo=ci(yk_b)[0], york_100_hi=ci(yk_b)[1],
               york_errors="per-hour NDROP" if d["relerr"] is not None else "constant")
    rows.append(row)
    print(f"  {src['label']:26s} {v.sum():6d} h  r={full['r'][0, i100]:5.2f}  binned {tab.S[i100]:5.2f}  OLS {full['ols'][0, i100]:5.2f}  "
          f"Deming {full['dem'][il0, 0, i100]:5.2f}  York {yk:5.2f}  RMA {full['rma'][0, i100]:5.2f}  [{row['check']}]")

res = pd.DataFrame(rows)
if MERGE_CAO:
    res["regime"] = res["regime"].replace({"Marine CAO": "Marine"})
res.to_csv(OUT_CSV, index=False, float_format="%.4f")
print(f"\nSaved {OUT_CSV} ({len(res)} sources)")

## 5. Table at 100 nm

`check = ok` means the hours and binned S(100 nm) reproduce the quicklook table. `ols − binned` is the 95% interval
of the paired difference over the same resampled days: an interval that excludes 0 means the binning itself moves S.

In [ ]:
EST = {"binned": ("binned medians (current)", "k", "o"), "ols": ("OLS, unbinned", "#1f77b4", "s"),
       "deming": ("Deming (λ per method)", "#2ca02c", "D"), "york": ("York (100 nm)", "#98df8a", "d"),
       "orth": ("orthogonal (λ = 1)", "#9467bd", "^"), "rma": ("RMA", "#d62728", "v"),
       "inv": ("inverse OLS", "0.6", "<"), "theilsen": ("Theil–Sen", "#ff7f0e", "x")}
cols = ["label", "method", "quality", "hours", "r100", "lam", "binned_100", "ols_100", "theilsen_100", "deming_100", "york_100",
        "orth_100", "rma_100", "inv_100", "ols_minus_binned_lo", "ols_minus_binned_hi", "check"]
with pd.option_context("display.width", 250, "display.max_rows", 100):
    display(res.sort_values(["regime", "binned_100"])[cols].round(2))

## 6. Forest plot: S(100 nm) by estimator

In [ ]:
r_ = res.sort_values(["regime", "binned_100"]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 0.32 * len(r_) + 1.5))
off = dict(zip(["binned", "ols", "theilsen", "deming", "york", "orth", "rma", "inv"], np.linspace(-0.3, 0.3, 8)))
for k, dy in off.items():
    lab, c, m = EST[k]
    yv = np.arange(len(r_)) + dy
    xv = r_[f"{k}_100"]
    if f"{k}_100_lo" in r_:
        ax.hlines(yv, r_[f"{k}_100_lo"], r_[f"{k}_100_hi"], color=c, lw=0.9, alpha=0.7)
    ax.scatter(xv, yv, marker=m, color=c, s=22, zorder=3, label=lab, edgecolor="k" if m not in "x" else None, linewidth=0.3)
ax.set_yticks(range(len(r_)))
ax.set_yticklabels([f"{l} ({g})" for l, g in zip(r_.label, r_.regime)], fontsize=8)
for tl, q in zip(ax.get_yticklabels(), r_.quality):
    tl.set_alpha(max(QA.get(q, 0.55), 0.45))
ax.axvline(0, color="0.5", lw=0.8); ax.axvline(1, color="0.5", lw=0.8, ls=":")
ax.set(xlabel="S(100 nm)", xlim=(-1.5, 3), ylim=(-0.7, len(r_) - 0.3),
       title="S(100 nm) by estimator (95% day-block CIs; inverse OLS can run off the axis when r is small)")
fig.legend(*ax.get_legend_handles_labels(), fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False)
ax.grid(axis="x", alpha=0.3)
fig.savefig(f"{FIG_DIR}/forest_s100.png", dpi=150, bbox_inches="tight")

## 7. Each estimator against the current recipe

Panel (d) shows why the estimators differ: the ratio of the binned slope to RMA tracks |r| the way OLS does by construction,
i.e. binning on x behaves like OLS and inherits its dilution.

In [ ]:
def regime_scatter(ax, x, y, df):
    for _, z in df.iterrows():
        ax.scatter(z[x] if isinstance(x, str) else x(z), z[y] if isinstance(y, str) else y(z), marker=MK[z.method],
                   color=REG_COL[z.regime], edgecolor="k", linewidth=0.5, s=40, alpha=QA.get(z.quality, 0.55))

fig, axs = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, k in zip(axs[:3], ["ols", "deming", "rma"]):
    regime_scatter(ax, "binned_100", f"{k}_100", res)
    ax.plot([-1, 3], [-1, 3], "k--", lw=0.8)
    ax.set(xlim=(-1, 2), ylim=(-1, 3), xlabel="binned medians S(100 nm)", ylabel=f"{EST[k][0]} S(100 nm)")
    ax.grid(alpha=0.3)
regime_scatter(axs[3], lambda z: abs(z.r100), lambda z: z.binned_100 / z.rma_100, res)
rr = np.linspace(0, 1, 50); axs[3].plot(rr, rr, "k--", lw=0.8, label="OLS/RMA = |r|")
axs[3].set(xlabel="|r| at 100 nm (hourly, unbinned)", ylabel="binned / RMA", xlim=(0, 1), ylim=(-0.5, 1.5)); axs[3].legend(fontsize=8)
axs[3].grid(alpha=0.3)
for a, l in zip(axs, "abcd"):
    a.set_title(f"({l})", loc="left")
h = ([Line2D([], [], ls="", marker="o", color=c, mec="k", label=k) for k, c in REG_COL.items() if k in set(res.regime)]
     + [Line2D([], [], ls="", marker=m, color="0.7", mec="k", label=k) for k, m in MK.items() if k in set(res.method)])
fig.legend(handles=h, loc="lower center", ncol=len(h), fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.06))
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/estimators_vs_binned.png", dpi=150, bbox_inches="tight")

## 8. Sensitivity to the error ratio λ (Deming)

λ = σ²(log Nd error)/σ²(log N(>D) error). λ → ∞ is OLS (all error in Nd), λ → 0 is inverse OLS (all error in N),
λ = 1 orthogonal. Markers: the λ used in the table (`SIG_Y`/`SIG_X`). Where the curve is flat the answer does not hinge on
the assumed errors; where it is steep (small r) S is not identified without knowing them.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for lab, (lams, s, lam0) in sweep.items():
    z = res.set_index("label").loc[lab]
    c, a = REG_COL[z.regime], QA.get(z.quality, 0.55)
    ax.plot(lams, s, "-", color=c, alpha=a, lw=1.2)
    ax.plot(lam0, np.interp(np.log(lam0), np.log(lams), s), MK[z.method], color=c, mec="k", ms=6, alpha=a)
ax.axhline(0, color="0.5", lw=0.8); ax.axhline(1, color="0.5", lw=0.8, ls=":")
ax.axvline(1, color="0.5", lw=0.8, ls="--")
ax.set(xscale="log", xlabel="λ = σ²(log Nd) / σ²(log N)   (left: error mostly in N, right: mostly in Nd → OLS)",
       ylabel="Deming S(100 nm)", ylim=(-1, 3), title="Deming slope vs assumed error ratio")
ax.legend(handles=[Line2D([], [], color=c, label=k) for k, c in REG_COL.items() if k in set(res.regime)], fontsize=8)
ax.grid(alpha=0.3)
fig.savefig(f"{FIG_DIR}/deming_lambda_sweep.png", dpi=150, bbox_inches="tight")

## 9. Sensitivity to the binning itself

Left: bin medians on x with 5–80 bins (the recipe uses ~15 h per bin, 5–20 bins). Middle: bin means instead of medians.
Right: bins on y (sorted on Nd) – the binned analogue of inverse OLS. If the left panel is flat, the bin count does not
matter; the gap between left and right is regression dilution in binned form.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for lab, b in binsens.items():
    z = res.set_index("label").loc[lab]
    for ax, k in zip(axs, ["x", "xmean", "y"]):
        ax.plot(NB_LIST, b[k], "-o", ms=3, color=REG_COL[z.regime], alpha=QA.get(z.quality, 0.55), lw=1)
for ax, t in zip(axs, ["bin medians, sorted on N(>100 nm)", "bin means, sorted on N(>100 nm)", "bin medians, sorted on Nd"]):
    ax.set(xscale="log", xticks=NB_LIST, xticklabels=NB_LIST, xlabel="number of bins", title=t)
    ax.axhline(0, color="0.5", lw=0.8); ax.axhline(1, color="0.5", lw=0.8, ls=":"); ax.grid(alpha=0.3)
axs[0].set(ylabel="S(100 nm)", ylim=(-1, 3))
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/binning_sensitivity.png", dpi=150, bbox_inches="tight")

## 10. S(D) by estimator for selected sources (band: 95% day-block CI)

In [ ]:
sel = [s for s in SEL if s in curves]
fig, axs = plt.subplots(1, len(sel), figsize=(4.2 * len(sel), 4), sharey=True, squeeze=False)
for ax, lab in zip(axs[0], sel):
    cv = curves[lab]
    for k in ["binned", "ols", "deming", "orth", "rma"]:
        pt, bs = cv["est"][k]
        ax.plot(cv["diam"], pt, color=EST[k][1], lw=1.6, label=EST[k][0])
        if k in ("binned", "ols", "deming"):
            lo, hi = np.nanpercentile(bs, [2.5, 97.5], axis=0)
            ax.fill_between(cv["diam"], lo, hi, color=EST[k][1], alpha=0.12, lw=0)
    if np.isfinite(cv["dd"]):
        ax.axvline(cv["dd"], color="0.3", ls="--", lw=0.8)
    ax.axhline(0, color="0.5", lw=0.8)
    ax.set(xscale="log", title=lab, xlabel="cutoff D (nm)", ylim=(-1, 3))
    ax.grid(alpha=0.3)
axs[0, 0].set_ylabel("S(D)"); axs[0, 0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/s_of_d_by_estimator.png", dpi=150, bbox_inches="tight")

## 11. The S(D$_\Delta$) vs D$_\Delta$ synthesis under each estimator

In [ ]:
k4 = ["binned", "ols", "deming", "rma"]
fig, axs = plt.subplots(2, 2, figsize=(13, 10), sharex=True, sharey=True)
rd = res[np.isfinite(res.D_Delta)]
for ax, k in zip(axs.flat, k4):
    for _, z in rd.iterrows():
        ye = [[max(z[f"{k}_DD"] - z[f"{k}_DD_lo"], 0)], [max(z[f"{k}_DD_hi"] - z[f"{k}_DD"], 0)]]
        ax.errorbar(z.D_Delta, z[f"{k}_DD"], yerr=ye, fmt=MK[z.method], color=REG_COL[z.regime], mec="k", mew=0.5,
                    ms=7, alpha=QA.get(z.quality, 0.55), elinewidth=0.8, capsize=2)
    med = rd[f"{k}_DD"].median()
    ax.axhline(0, color="0.5", lw=0.8); ax.axhline(1, color="0.5", lw=0.8, ls=":")
    ax.set(xscale="log", title=f"{EST[k][0]}: median S(D$_\\Delta$) = {med:.2f}")
    ax.grid(alpha=0.3)
for ax in axs[1]:
    ax.set_xlabel("D$_\\Delta$ (nm)")
for ax in axs[:, 0]:
    ax.set_ylabel("S(D$_\\Delta$)")
axs[0, 0].set_ylim(-1.2, 3); axs[0, 0].set_xlim(12, 400)
from matplotlib.ticker import FixedLocator, ScalarFormatter, NullFormatter
for ax in axs.flat:
    ax.xaxis.set_major_locator(FixedLocator([20, 30, 50, 70, 100, 150, 200, 300]))
    ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_formatter(NullFormatter())
h = ([Line2D([], [], ls="", marker="o", color=c, mec="k", label=k) for k, c in REG_COL.items() if k in set(rd.regime)]
     + [Line2D([], [], ls="", marker=m, color="0.7", mec="k", label=k) for k, m in MK.items() if k in set(rd.method)])
fig.legend(handles=h, loc="lower center", ncol=len(h), fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.03))
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/s_at_ddelta_by_estimator.png", dpi=150, bbox_inches="tight")

## 12. Regime medians by estimator (robust and moderate sources only, then all)

In [ ]:
ks = ["binned", "ols", "theilsen", "deming", "york", "orth", "rma", "inv"]
for title, sub in [("robust + moderate", res[res.quality.isin(["robust", "moderate"])]), ("all", res)]:
    s100 = sub.groupby("regime")[[f"{k}_100" for k in ks]].median()
    sdd = sub.groupby("regime")[[f"{k}_DD" for k in ["binned", "ols", "deming", "orth", "rma", "inv"]]].median()
    s100.insert(0, "n", sub.groupby("regime").size())
    print(f"\nS(100 nm), regime medians, {title}:"); display(s100.round(2))
    print(f"S(D_Delta), regime medians, {title}:"); display(sdd.round(2))

## Reading the results

* **Binned ≈ OLS** (panel 7a on the 1:1 line; section 9 flat in the bin count) → the binning is not what sets S; the
  choice that matters is which variable is treated as error-free.
* **Deming/York well above binned at sources with low r** → the synthesis S is a dilution-biased lower bound there.
  How far above depends on λ (section 8); the honest range is between the OLS and the Deming value for plausible λ.
* **Aircraft (high r, small errors in both)** should show little spread between estimators – a useful anchor.
* For the model comparison, the model S was computed with the same binned recipe on grid-cell output where x has no
  measurement error, so the like-for-like obs number is the dilution-corrected one only if the model's hourly x and y
  carry comparable unshared variance. Worth stating explicitly when comparing.